# Grid2Op HF Jobs Training Notebook

This notebook is the organizer-facing training entrypoint for the repository.

It is designed to:
- run from the checked-out codebase
- validate the required HF bucket assets before launch
- launch the main RL training job on Hugging Face Jobs
- use `HF_TOKEN` and `WANDB_API_KEY` as explicit configuration inputs

The primary training path in this notebook is **HF Jobs on `l40sx1`**.


## Configuration Guide

Optional `.env` placement:
- place `.env` at the repository root
- example path: `grid2op-openenv/.env`

Useful keys:
```env
HF_TOKEN=your_hf_token
WANDB_API_KEY=your_wandb_key
HF_USERNAME=Sidharth1743
GRID2OP_DATASET_PATH=/mnt/datasets/ieee118_teacher_actions_v1.jsonl
GRID2OP_SFT_ADAPTER=Sidharth1743/grid2op-qwen3-4b-sft-final
GRID2OP_GRPO_OUTPUT=/mnt/runs/grid2op-qwen3-4b-grpo-ieee118-v1
GRID2OP_GRPO_RUN_NAME=grid2op-qwen3-4b-grpo-ieee118-v1
```

For HF Jobs, `HF_TOKEN` is required. `WANDB_API_KEY` is strongly recommended.


In [ ]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()
repo_dir = cwd if (cwd / 'scripts').exists() else cwd.parent
assert (repo_dir / 'scripts').exists(), f'Could not find repo root from {cwd}'
os.chdir(repo_dir)
print({'repo_dir': str(repo_dir)})


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
from pathlib import Path
uv_bin_dir = Path.home() / '.local' / 'bin'
if uv_bin_dir.exists():
    os.environ['PATH'] = f"{uv_bin_dir}:{os.environ['PATH']}"
!command -v uv || echo 'uv already available in PATH or installed elsewhere'


In [ ]:
!UV_CACHE_DIR=/tmp/uv-cache uv sync --no-dev
!UV_CACHE_DIR=/tmp/uv-cache uv pip install 'huggingface_hub>=1.8.0' wandb


In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os
import subprocess

load_dotenv(repo_dir / '.env', override=False)

HF_TOKEN = os.environ.get('HF_TOKEN', '')
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')
HF_USERNAME = os.environ.get('HF_USERNAME', 'Sidharth1743')
GIT_REPO_URL = os.environ.get('GRID2OP_GIT_REPO_URL', '')

try:
    CURRENT_BRANCH = subprocess.check_output(
        ['git', 'rev-parse', '--abbrev-ref', 'HEAD'],
        cwd=repo_dir,
        text=True,
    ).strip()
except Exception:
    CURRENT_BRANCH = 'ieee118-port'
if not CURRENT_BRANCH or CURRENT_BRANCH == 'HEAD':
    CURRENT_BRANCH = 'ieee118-port'

try:
    CURRENT_COMMIT = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'],
        cwd=repo_dir,
        text=True,
    ).strip()
except Exception:
    CURRENT_COMMIT = ''

if not GIT_REPO_URL:
    try:
        GIT_REPO_URL = subprocess.check_output(
            ['git', 'config', '--get', 'remote.origin.url'],
            cwd=repo_dir,
            text=True,
        ).strip()
    except Exception:
        GIT_REPO_URL = 'https://github.com/Sidharth1743/grid2op-openenv.git'

# HF bucket-mounted paths used by the HF Job itself.
HF_DATASET_PATH = os.environ.get('GRID2OP_DATASET_PATH', '/mnt/datasets/ieee118_teacher_actions_v1.jsonl')
HF_SFT_ADAPTER = os.environ.get('GRID2OP_SFT_ADAPTER', 'Sidharth1743/grid2op-qwen3-4b-sft-final')
HF_GRPO_OUTPUT = os.environ.get('GRID2OP_GRPO_OUTPUT', '/mnt/runs/grid2op-qwen3-4b-grpo-ieee118-v1')
HF_GRPO_RUN_NAME = os.environ.get('GRID2OP_GRPO_RUN_NAME', 'grid2op-qwen3-4b-grpo-ieee118-v1')
HF_JOB_FLAVOR = os.environ.get('HF_JOB_FLAVOR', 'l40sx1')
HF_BUCKET_REPO = os.environ.get('GRID2OP_BUCKET_REPO', f'{HF_USERNAME}/grid2op-finals')

print({
    'hf_token_set': bool(HF_TOKEN),
    'wandb_api_key_set': bool(WANDB_API_KEY),
    'hf_username': HF_USERNAME,
    'current_branch': CURRENT_BRANCH,
    'current_commit': CURRENT_COMMIT,
    'git_repo_url': GIT_REPO_URL,
    'hf_bucket_repo': HF_BUCKET_REPO,
    'hf_dataset_path': HF_DATASET_PATH,
    'hf_sft_adapter': HF_SFT_ADAPTER,
    'hf_grpo_output': HF_GRPO_OUTPUT,
    'hf_grpo_run_name': HF_GRPO_RUN_NAME,
    'hf_job_flavor': HF_JOB_FLAVOR,
})


## HF Bucket Asset Preflight

This cell verifies that the dataset exists in the HF bucket and that the SFT adapter is reachable either as a bucket path or as a public Hugging Face model repo. It does not require any local dataset file.


In [ ]:
from huggingface_hub import HfApi

if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required for HF bucket preflight checks.')

api = HfApi(token=HF_TOKEN)
repo_files = api.list_repo_files(repo_id=HF_BUCKET_REPO)
dataset_repo_path = HF_DATASET_PATH.removeprefix('/mnt/')
adapter_is_bucket_path = HF_SFT_ADAPTER.startswith('/mnt/')

dataset_exists = dataset_repo_path in repo_files
if adapter_is_bucket_path:
    adapter_repo_prefix = HF_SFT_ADAPTER.removeprefix('/mnt/').rstrip('/')
    adapter_exists = any(path == adapter_repo_prefix or path.startswith(adapter_repo_prefix + '/') for path in repo_files)
else:
    adapter_repo_prefix = HF_SFT_ADAPTER
    try:
        api.model_info(HF_SFT_ADAPTER)
        adapter_exists = True
    except Exception:
        adapter_exists = False

print({
    'bucket_repo': HF_BUCKET_REPO,
    'dataset_repo_path': dataset_repo_path,
    'dataset_exists': dataset_exists,
    'adapter_is_bucket_path': adapter_is_bucket_path,
    'adapter_repo_prefix': adapter_repo_prefix,
    'adapter_exists': adapter_exists,
})

if not dataset_exists:
    raise FileNotFoundError(f'Missing dataset in HF bucket: {dataset_repo_path}')
if not adapter_exists:
    raise FileNotFoundError(f'Missing SFT adapter at: {adapter_repo_prefix}')


## Main RL Training Launch (HF Jobs, L40S)

This is the primary notebook action. It launches the IEEE 118 GRPO/DAPO job on Hugging Face Jobs using the existing project launcher.


In [ ]:
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required to launch the HF Job from this notebook.')
if not WANDB_API_KEY:
    raise RuntimeError('WANDB_API_KEY is recommended and currently required by the HF GRPO launcher.')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['HF_USERNAME'] = HF_USERNAME
os.environ['BRANCH_NAME'] = CURRENT_BRANCH
os.environ['GIT_REF'] = CURRENT_COMMIT
os.environ['GIT_REPO_URL'] = GIT_REPO_URL
os.environ['HF_JOB_FLAVOR'] = HF_JOB_FLAVOR
os.environ['DATASET_PATH'] = HF_DATASET_PATH
os.environ['ADAPTER_PATH'] = HF_SFT_ADAPTER
os.environ['RUN_NAME'] = HF_GRPO_RUN_NAME
os.environ['OUTPUT_PATH'] = HF_GRPO_OUTPUT
os.environ['FOLLOW_HF_JOB_LOGS'] = 'true'

print({
    'HF_USERNAME': os.environ['HF_USERNAME'],
    'BRANCH_NAME': os.environ['BRANCH_NAME'],
    'GIT_REF': os.environ['GIT_REF'],
    'GIT_REPO_URL': os.environ['GIT_REPO_URL'],
    'HF_JOB_FLAVOR': os.environ['HF_JOB_FLAVOR'],
    'DATASET_PATH': os.environ['DATASET_PATH'],
    'ADAPTER_PATH': os.environ['ADAPTER_PATH'],
    'RUN_NAME': os.environ['RUN_NAME'],
    'OUTPUT_PATH': os.environ['OUTPUT_PATH'],
    'FOLLOW_HF_JOB_LOGS': os.environ['FOLLOW_HF_JOB_LOGS'],
})


In [ ]:
import subprocess

subprocess.run(
    ['bash', '-lc', 'bash scripts/run_hf_ieee118_grpo.sh'],
    check=True,
)


## Notes

- This notebook launches the real training job on HF Jobs, not on the local runtime.
- The configured GPU flavor is `l40sx1` by default.
- The notebook validates the dataset in the HF bucket and validates the SFT adapter either in the HF bucket or as a public model repo.
- The HF Job reads from the bucket-mounted path such as `/mnt/datasets/ieee118_teacher_actions_v1.jsonl`.
- The HF Job is pinned to the current git commit when that commit hash is available.
- The default SFT adapter is the public model repo `Sidharth1743/grid2op-qwen3-4b-sft-final`.
